In [ ]:
%pylab inline
import scipy as sc
from scipy import optimize
from copy import copy
import eucare as ec

In [ ]:
def kawasaki_sum(v):
    angles = np.abs(np.array([e['in_angle'] for e in v.incoming_iter()]))
    assert len(angles) % 2 == 0
    return np.sum(angles * (-1) ** np.arange(len(angles)))
    
def max_kawasaki_sum(vertices):
    if isinstance(vertices, ec.half.HalfEdgeGraph):
        vertices = [v for v in vertices.vertices if not v.on_border()]
    return np.max([kawasaki_sum(v) for v in vertices])

In [ ]:
render_settings = dict(
    figsize=(10, 10),
    render_edges=True,
    render_faces=False,
    render_vertices=False,
)

In [ ]:
def fancy_graph():
    hextile0 = ec.prototiles.RegularEuclideanTile(6, edge_labels=[0]*6)
    hextile1 = ec.prototiles.RegularEuclideanTile(6, edge_labels=[0]*6)
    tritile = ec.prototiles.RegularEuclideanTile(3, edge_labels=[0]*3)
    ec.example_tilesets.align_tiles(hextile0, 0, hextile1, 0)
    ec.example_tilesets.align_tiles(hextile1, 0, tritile, 0)
    ec.example_tilesets.align_tiles(tritile, 0, tritile, 0)

    G = ec.example_graphs.from_tiles([tritile, hextile1], 3)
    #G = ec.example_graphs.from_tiles(ec.example_tilesets.platonic(3), 5)
    #G = ec.example_graphs.from_tiles(ec.example_tilesets.t_3_3_4_3_4(), 4)
    ec.colorization.congruency_colorize(G)
    #G.show(**render_settings)

    #G = ec.conway.gyro_graph()(G)
    G = ec.conway.join_graph()(G)
    #G = ec.conway.dual_graph()(G)
    ec.colorization.congruency_colorize(G)
    #G.show(**render_settings)

    ps, vs = G.get_position_view()

    k = ps.copy()
    k = np.array([complex(*ki) for ki in k])

    #k -= complex(*platonic(3)[0].points[-1])
    #k = k**2 / 3
    #k = 5 * k

    k = np.stack([k.real, k.imag], axis=-1)
    #k[:, 0] *= 0.5

    ps[:] = k
    G.recompute_lengths_and_angles()
    return G
fancy_graph().show(**render_settings)

In [ ]:
G = fancy_graph()

# Step 1: Choose direction for every interior edge.
def random_directed_set(edges):
    if isinstance(edges, ec.half.HalfEdgeGraph):
        edges = edges.halfedges
    directed_edges = set()
    for e in edges:
        if e.rev not in directed_edges:
            directed_edges.add(e)
    return directed_edges

print(len(G.halfedges))
directed_edges = random_directed_set([e for e in G.halfedges 
                                      if not (e.on_border() or e.rev.on_border())])
print(f'n edges: {len(directed_edges)}')

# Step 2: Construct array of all vectors of the directed edges, mapping from edge to index
edge_vectors = np.stack([e.orig['pos'] - e.dest['pos'] for e in directed_edges])

dual_vectors = edge_vectors @ ec.base.rotation_matrix(np.pi/2)
dual_directions = dual_vectors / np.linalg.norm(dual_vectors, axis=1, keepdims=True)
edges_to_ids = {e: i for i, e in enumerate(directed_edges)}
print('dual directions shape:', dual_directions.shape)

# Step 3: Formulate constraints as linear problem Ax = 0
# every constraint is a row in the matrix A. Every interior vertex leads to a constraint. Hence, compute one row for each interior vertex.

interior_vertices = [v for v in G.vertices if not v.on_border()]
print('number of interor vertices=constraints:', len(interior_vertices))

rows = []
n_edges = len(directed_edges)
for v in interior_vertices:
    row = np.zeros(n_edges, dtype=np.float32)
    for e in v.outgoing_iter():
        if e in directed_edges:
            row[edges_to_ids[e]] = -1
        else:
            row[edges_to_ids[e.rev]] = 1
    rows.append(row)
B = np.stack(rows)
A = (B[:, None, :] * dual_directions.T[: None]).reshape(-1, n_edges)
print('A.shape:', A.shape)
U = sc.linalg.null_space(A)
print('U.shape:', U.shape)
assert U.shape[1] > 0, f'G does not have a reciprocal figure!'
# Step 4: Formulate and solve least squares problem to make reciprocal graph as 
# similar as possible to result of conway.dual_graph()(G)

# need map face in primal -> vertex in dual!

# need linear map coords in solution space -> dual edge lenghts
# this is just U @ coords
#coords = np.random.rand(U.shape[1])
#print(((U @ coords)[:, None] * dual_directions).shape)

# need linear map dual edge lengths -> dual edge offsets
# this is just dual_directions

# need linear map (dual edge offsets, position of interior_vertices[0]) -> dual vertex positons
to_process = set(G.faces)
anchor = to_process.pop()
coefficients = {anchor: np.zeros(n_edges, dtype=np.float32)}
border = {anchor}
while border:
    new_border = set()
    for f in border:
        for e in f.halfedge_iter():
            f2 = e.rev.face
            if f2 not in coefficients:
                if e in directed_edges:
                    coefficients[f2] = copy(coefficients[f])
                    coefficients[f2][edges_to_ids[e]] = -1
                elif e.rev in directed_edges:
                    coefficients[f2] = copy(coefficients[f])
                    coefficients[f2][edges_to_ids[e.rev]] = 1
                else:
                    continue
                new_border.add(f2)
    border = new_border
assert set(coefficients.keys()) == set(G.faces)

faces = G.faces
n_faces = len(faces)
print('n_faces:', n_faces)
D2P = np.stack([coefficients[f] for f in faces])
print('D2P.shape:', D2P.shape)

M = (D2P @ U)#[:, None] * dual_directions

M = np.moveaxis(np.dot(D2P, np.moveaxis(U[:, :, None] * dual_directions[:, None], 0, 1)), 1, 2)
print('M.shape', M.shape)

# Add two columns to M, corresponding to the offset of the dual graph
xy_columns = np.zeros((n_faces, 2, 2), dtype=np.float32)
xy_columns[:, 0, 0] = 1
xy_columns[:, 1, 1] = 1
M = np.concatenate([xy_columns, M], axis=-1)

# Get 'ground truth' face centers: for now just com of the faces
face_centers = np.stack([f.midpoint() for f in faces])
print('face_centers.shape', face_centers.shape)

# flatten xy
M = M.reshape(n_faces * 2, -1)
face_centers = face_centers.reshape(n_faces * 2)

# solve the least squares problem
sol = sc.optimize.lsq_linear(M, face_centers)
assert sol['success'], f"{sol['message']}"
sol = sol['x']
dual_vertices = M @ sol
dual_vertices = dual_vertices.reshape(-1, 2)

#plt.figure(figsize=(5, 5))
#plt.scatter(dual_vertices[:, 0], dual_vertices[:, 1])
#from eucare import plotting
#plotting.set_equal_aspect()
#plt.show()

# Step 5: make reciprocal figure into face graph
D, (_, _, f_map) = G.copy(return_mappings=True)
f2p = {f_map[f]: dual_vertices[i] for i, f in enumerate(faces)}
D = ec.conway.dual_graph()(D)
for v in D.vertices:
    v['pos'] = f2p[v['pre_conway']]

# Step 6: get shrink-rotate graph and apply mapping
SRG, (_, _, f_map) = G.copy(return_mappings=True)
f2p = {f_map[f]: dual_vertices[i] for i, f in enumerate(faces)}
SRG = ec.conway.twist_rotate_graph()(SRG)

alpha, factor = np.pi/9, 0.7
for f in filter(lambda f: 'twistrotate' in f.attributes, SRG.faces):
    ps, vs = np.array([[v['pos'], v] for v in f.vertex_iter()]).T
    ps = np.stack(ps)
    
    midpoint = np.mean(ps, axis=0, keepdims=True)
    ps = midpoint + (ps - midpoint) * 2
    
    rotation_center = f2p[f['pre_conway']]
    ps = rotation_center + (ps - rotation_center) @ ec.base.rotation_matrix(alpha) * factor
    
    #midpoint = np.mean(ps, axis=0, keepdims=True)
    #ps = midpoint + (ps - midpoint) * 2 * factor
    
    
    for v, p in zip(vs, ps):
        v['pos'] = p
        
SRG.recompute_lengths_and_angles()
ec.colorization.congruency_colorize(SRG)
SRG.show(**render_settings)
print(max_kawasaki_sum(SRG))

D.add_graph(G)
D.show(**render_settings)